# Batch processing — reusing previously saved parameters

When you have already analyzed a folder, every video has an `Analysis_<video_name>/<video_name>_analysis_parameters.csv` next to it. Passing `load_params=True` to `BatchExperiment` makes pyKrait read that file per video and use the exact same parameters as before — useful for reproducing prior results, re-running after a code change, or re-using saved Cellpose masks.

If a video has no saved parameter file, the constructor falls back to the `params` argument and emits a warning.

In [ ]:
from pykrait.pipeline.pipeline import BatchExperiment, AnalysisParameters
from pykrait.io.files import concat_analysis_files

## 1. Point at the folder

Use the same folder you ran previously. The parameter files will be discovered automatically.

In [ ]:
folder = "/path/to/your/videos"
extension = ".czi"  # also supports ".tif" / ".tiff"

## 2. Provide a fallback parameter set

These parameters are only used for videos whose `*_analysis_parameters.csv` is missing or fails to parse. For videos with a valid file, the saved values take precedence.

In [ ]:
fallback_params = AnalysisParameters(
    cellpose_model_path="cpsam",
)

## 3. Run with `load_params=True`

Because the previous run already produced segmentation masks and raw intensities, you can skip Cellpose and intensity extraction to re-run only the downstream analysis steps. Set them to `False` if you want a full reanalysis from scratch.

In [ ]:
experiment = BatchExperiment(
    folder=folder,
    params=fallback_params,
    extension=extension,
    load_params=True,
)

experiment.run(
    skip_cellpose=True,
    skip_intensity_extraction=True,
)

## 4. Refresh the overview CSV

Re-runs overwrite each video's per-result CSVs. Regenerate the folder-level overview to reflect the new analysis.

In [ ]:
concat_analysis_files(folder, filetype="output")
concat_analysis_files(folder, filetype="parameters")